# None and Optional Values

Handling missing values is one of the most common sources of bugs. Let's master it.

## What Is None?

In [ ]:
# None is Python's null/nothing value
x = None

print(f"Value: {x}")
print(f"Type: {type(x)}")
print(f"Boolean: {bool(x)}")  # None is falsy

In [ ]:
# None is a singleton
a = None
b = None

print(f"a is b: {a is b}")  # True - same object
print(f"id(a) == id(b): {id(a) == id(b)}")

## Checking for None

In [ ]:
value = None

# CORRECT: Use 'is' for None
if value is None:
    print("Value is None")

if value is not None:
    print("Value exists")

# WRONG: Don't use == for None (works but bad style)
# if value == None:  # Don't do this!

In [ ]:
# Why 'is' matters
class WeirdClass:
    def __eq__(self, other):
        return True  # Claims to equal everything!

weird = WeirdClass()
print(f"weird == None: {weird == None}")  # True (wrong!)
print(f"weird is None: {weird is None}")  # False (correct!)

## None vs Falsy Values

In [ ]:
# These are all falsy but different!
values = [None, False, 0, "", [], {}, set()]

for val in values:
    is_none = val is None
    is_falsy = not val
    print(f"{repr(val):10} | is None: {is_none} | is falsy: {is_falsy}")

In [ ]:
# Common bug: treating 0 or "" as None
def process(value):
    # BUG: This treats 0 and "" as missing!
    if not value:
        return "missing"
    return f"got: {value}"

print(process(None))  # "missing" - correct
print(process(0))     # "missing" - probably wrong!
print(process(""))    # "missing" - probably wrong!

In [ ]:
# FIXED: Check specifically for None
def process_fixed(value):
    if value is None:
        return "missing"
    return f"got: {value}"

print(process_fixed(None))  # "missing"
print(process_fixed(0))     # "got: 0"
print(process_fixed(""))    # "got: "

## Default Values

In [ ]:
# Pattern 1: Using 'or' for defaults (careful!)
name = "" or "Anonymous"  # Gets "Anonymous"!
count = 0 or 10           # Gets 10!

print(f"Name: {name}")
print(f"Count: {count}")

# 'or' returns first truthy value, or last value
# Only safe when 0, "", etc. should also use default

In [ ]:
# Pattern 2: Ternary for None-only default
value = 0
result = value if value is not None else 10
print(f"Result: {result}")  # 0, not 10

value = None
result = value if value is not None else 10
print(f"Result: {result}")  # 10

In [ ]:
# Pattern 3: dict.get() with default
config = {"timeout": 0, "debug": False}  # Note: 0 and False are valid values!

# .get() returns None if key missing
timeout = config.get("timeout")
retries = config.get("retries")  # None - key doesn't exist

print(f"Timeout: {timeout}")
print(f"Retries: {retries}")

# .get() with default
retries = config.get("retries", 3)
print(f"Retries with default: {retries}")

## Function Parameter Defaults

In [ ]:
# DANGER: Mutable default argument!
def bad_append(item, items=[]):
    items.append(item)
    return items

print(bad_append(1))  # [1]
print(bad_append(2))  # [1, 2] - Bug! Same list!
print(bad_append(3))  # [1, 2, 3] - Getting worse!

In [ ]:
# CORRECT: Use None as sentinel
def good_append(item, items=None):
    if items is None:
        items = []  # New list each call
    items.append(item)
    return items

print(good_append(1))  # [1]
print(good_append(2))  # [2] - Fresh list!
print(good_append(3))  # [3] - Fresh list!

In [ ]:
# Common pattern in AI code
def create_config(timeout=None, headers=None, options=None):
    """Create config with defaults for mutable types."""
    return {
        "timeout": timeout if timeout is not None else 30,
        "headers": headers if headers is not None else {},
        "options": options if options is not None else [],
    }

config1 = create_config()
config2 = create_config()

config1["headers"]["auth"] = "token"
print(f"Config1 headers: {config1['headers']}")
print(f"Config2 headers: {config2['headers']}")  # Empty - not shared!

## Optional Type Hints

In [ ]:
from typing import Optional

# Optional[X] means X | None

def find_user(user_id: int) -> Optional[str]:
    """Find user by ID. Returns None if not found."""
    users = {1: "Alice", 2: "Bob"}
    return users.get(user_id)

# Python 3.10+ can use X | None instead
def find_user_modern(user_id: int) -> str | None:
    users = {1: "Alice", 2: "Bob"}
    return users.get(user_id)

print(find_user(1))   # "Alice"
print(find_user(99))  # None

## Safe Navigation Patterns

In [ ]:
# Pattern 1: Guard clauses
def get_user_email(user):
    if user is None:
        return None
    if "email" not in user:
        return None
    return user["email"]

print(get_user_email(None))
print(get_user_email({}))
print(get_user_email({"email": "test@example.com"}))

In [ ]:
# Pattern 2: try/except for nested access
data = {
    "user": {
        "profile": {
            "email": "alice@example.com"
        }
    }
}

def safe_get(data, *keys, default=None):
    """Safely navigate nested dict."""
    try:
        result = data
        for key in keys:
            result = result[key]
        return result
    except (KeyError, TypeError):
        return default

print(safe_get(data, "user", "profile", "email"))
print(safe_get(data, "user", "profile", "phone"))
print(safe_get(data, "user", "missing", "email"))

In [ ]:
# Pattern 3: getattr with default for objects
class User:
    def __init__(self, name):
        self.name = name

user = User("Alice")

name = getattr(user, "name", "Unknown")
email = getattr(user, "email", "no-email")

print(f"Name: {name}")
print(f"Email: {email}")

## Common AI Code Patterns

In [ ]:
# Pattern 1: Optional chaining simulation
def get_nested(obj, *attrs, default=None):
    """Get nested attribute safely."""
    try:
        for attr in attrs:
            obj = getattr(obj, attr)
        return obj
    except AttributeError:
        return default

# Usage: get_nested(user, 'profile', 'settings', 'theme')

In [ ]:
# Pattern 2: Filter out None values
values = [1, None, 2, None, 3, None]

# Filter None
non_none = [v for v in values if v is not None]
print(f"Without None: {non_none}")

# Or using filter
non_none = list(filter(lambda x: x is not None, values))
print(f"Using filter: {non_none}")

In [ ]:
# Pattern 3: First non-None value
def coalesce(*values):
    """Return first non-None value."""
    for v in values:
        if v is not None:
            return v
    return None

result = coalesce(None, None, "default", "other")
print(f"First non-None: {result}")

In [ ]:
# Pattern 4: Null object pattern
class NullUser:
    """Null object that safely handles missing user."""
    name = "Anonymous"
    email = "no-email"
    
    def __bool__(self):
        return False

def get_user(user_id):
    users = {1: type('User', (), {'name': 'Alice', 'email': 'alice@example.com'})()}
    return users.get(user_id) or NullUser()

user = get_user(1)
print(f"User 1: {user.name}")

user = get_user(99)
print(f"User 99: {user.name}")  # Safe - returns NullUser

## Summary

| Pattern | Use Case |
|---------|----------|
| `x is None` | Check if None |
| `x is not None` | Check if not None |
| `x or default` | Default for any falsy |
| `x if x is not None else default` | Default only for None |
| `dict.get(key, default)` | Safe dict access |
| `getattr(obj, attr, default)` | Safe attribute access |
| `Optional[X]` / `X \| None` | Type hint for nullable |

## Module Complete!

You now understand:
- Core collections (list, dict, set, tuple)
- Specialized collections (Counter, defaultdict, deque)
- Strings and bytes
- Handling None and optional values

Next module: Python Magic!